<a href="https://colab.research.google.com/github/marioluciofjr/codesoiram.python/blob/main/c2pa_caseiro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# CÉLULA 1: INSTALAÇÃO DAS FERRAMENTAS (PIPS)
# ==============================================================================
# @title 1. Instalar Bibliotecas
# Esta etapa baixa as ferramentas necessárias para o código funcionar.
# - c2pa-python: Ferramenta oficial para criar as "Content Credentials".
# - cryptography: Ferramenta para criar as chaves de segurança digital.
# - gradio: Ferramenta para criar a tela visual (botões e janelas).

!pip install -q c2pa-python cryptography gradio
print("✅ Ferramentas instaladas com sucesso!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 47.9 MB/s eta 0:00:00
✅ Ferramentas instaladas com sucesso!


In [16]:
# ==============================================================================
# CÉLULA 2: IMPORTAÇÃO DOS MÓDULOS (IMPORTS)
# ==============================================================================
# @title 2. Carregar Módulos
# Aqui nós "ligamos" as ferramentas que acabamos de instalar para começar a usar.

import os
import json
import datetime
import urllib.request
import c2pa

# Ferramentas de Criptografia (Segurança)
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.backends import default_backend
from cryptography import x509
from cryptography.x509.oid import NameOID

# Ferramentas do Google Colab (Upload/Download)
from google.colab import files
from IPython.display import display, Markdown

# Ferramenta Visual
import gradio as gr
print("✅ Módulos carregados e prontos para uso.")

✅ Módulos carregados e prontos para uso.


In [17]:
# ==============================================================================
# CÉLULA 3: VARIÁVEIS GLOBAIS (CONFIGURAÇÕES)
# ==============================================================================
# @title 3. Configurações Globais
# Aqui definimos regras que valem para todo o código.
# 1. Versão atual da ferramenta C2PA
VERSAO_C2PA = c2pa.sdk_version()
# 2. Configurações do Usuário (Edite ou Execute para digitar)
# Ao rodar esta célula, você poderá definir o nome e a validade.
print(">>> CONFIGURAÇÕES GLOBAIS <<<")
NOME_AUTOR_GLOBAL = input("Digite seu Nome Completo (Padrão: Anônimo): ").strip()
if not NOME_AUTOR_GLOBAL: NOME_AUTOR_GLOBAL = "Anônimo"
# Validade do Certificado
try:
    dias_input = input("Validade do Certificado em dias (Padrão: 30): ").strip()
    VALIDADE_DIAS = int(dias_input) if dias_input else 30
except:
    VALIDADE_DIAS = 30
print(f"\n✅ Autor Definido: {NOME_AUTOR_GLOBAL}")
print(f"✅ Validade Definida: {VALIDADE_DIAS} dias")
# 3. Mapa dos formatos suportados
# Verificação de compatibilidade (Debug):
# AVI -> video/msvideo (OK)
# AVIF -> image/avif (OK)
# PDF -> application/pdf (OK)
# etc.
MAPA_MIMES = {
    # Imagens
    '.jpg': 'image/jpeg',
    '.jpeg': 'image/jpeg',
    '.png': 'image/png',
    '.webp': 'image/webp',
    '.gif': 'image/gif',
    '.avif': 'image/avif',
    '.heic': 'image/heic',
    '.heif': 'image/heif',
    '.dng': 'image/x-adobe-dng',
    '.svg': 'image/svg+xml',
    '.tif': 'image/tiff',
    '.tiff': 'image/tiff',
    '.c2pa': 'application/x-c2pa-manifest-store',

    # Vídeos
    '.mp4': 'video/mp4',
    '.mov': 'video/quicktime',
    '.avi': 'video/msvideo',

    # Áudios
    '.mp3': 'audio/mpeg',
    '.wav': 'audio/wav',
    '.m4a': 'audio/mp4'
}
print(f"✅ Suporte ativo para {len(MAPA_MIMES)} formatos.")

>>> CONFIGURAÇÕES GLOBAIS <<<
Digite seu Nome Completo (Padrão: Anônimo): Mário Lúcio
Validade do Certificado em dias (Padrão: 30): 45

✅ Autor Definido: Mário Lúcio
✅ Validade Definida: 45 dias
✅ Suporte ativo para 19 formatos.


In [18]:
# ==============================================================================
# CÉLULA 4: CREDENCIAIS DE SEGURANÇA (Função Criptografada)
# ==============================================================================
# @title 4. Criar Chaves e Certificados (PKI)
# Para assinar um arquivo digitalmente, precisamos de uma "Identidade Digital".
# Como não temos uma identidade da Adobe (que custa dinheiro e verificação),
# vamos criar uma "Identidade Local" válida e segura.
#
# Processo Técnico:
# 1. Criamos uma "Autoridade Raiz" (Root CA) - O Chefe da Segurança.
# 2. Criamos um "Crachá de Uso" (Leaf Cert) - A Permissão para Assinar.
# 3. Fazemos o sistema confiar na nossa Autoridade Raiz.
def criar_autoridade_certificadora():
    """Cria a Autoridade Suprema (Root CA) que validará nossos documentos."""
    print("   -> Gerando Autoridade Certificadora (Root CA)...")

    # Cria a chave secreta da autoridade
    chave_privada = ec.generate_private_key(ec.SECP256R1(), default_backend())

    # Define quem é a autoridade
    identidade = x509.Name([
        x509.NameAttribute(NameOID.COUNTRY_NAME, u"BR"), # Brasil
        x509.NameAttribute(NameOID.ORGANIZATION_NAME, u"Google Colab"),
        x509.NameAttribute(NameOID.COMMON_NAME, u"Colab PKI Root"),
    ])

    # Cria o "Identifier" (Identidade técnica)
    public_key_bytes = chave_privada.public_key().public_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PublicFormat.SubjectPublicKeyInfo
    )
    ski = x509.SubjectKeyIdentifier.from_public_key(chave_privada.public_key())

    # Gera o certificado com validade de 1 ano
    certificado = x509.CertificateBuilder().subject_name(
        identidade
    ).issuer_name(
        identidade # Ela assina a si mesma (é a raiz)
    ).public_key(
        chave_privada.public_key()
    ).serial_number(
        x509.random_serial_number()
    ).not_valid_before(
        datetime.datetime.utcnow()
    ).not_valid_after(
        datetime.datetime.utcnow() + datetime.timedelta(days=365)
    ).add_extension(
        x509.BasicConstraints(ca=True, path_length=0), critical=True, # É uma autoridade
    ).add_extension(
        ski, critical=False
    ).add_extension(
        x509.KeyUsage( # Permissões dela
            digital_signature=False,
            key_cert_sign=True, # Pode assinar outros certificados
            crl_sign=True,
            key_encipherment=False,
            content_commitment=False,
            data_encipherment=False,
            key_agreement=False,
            encipher_only=False,
            decipher_only=False
        ), critical=True
    ).sign(chave_privada, hashes.SHA256(), default_backend())

    return chave_privada, certificado
def criar_certificado_usuario(ca_privada, ca_cert):
    """Cria o Crachá do Usuário (Leaf) assinado pela Autoridade."""
    print("   -> Gerando Certificado do Usuário (Leaf)...")

    chave_usuario = ec.generate_private_key(ec.SECP256R1(), default_backend())

    identidade = x509.Name([
        x509.NameAttribute(NameOID.COUNTRY_NAME, u"BR"),
        x509.NameAttribute(NameOID.ORGANIZATION_NAME, u"Google Colab"),
        x509.NameAttribute(NameOID.COMMON_NAME, u"Google Colab com Antigravity"), # Nome do Dispositivo
    ])

    # Identificadores técnicos para ligar este crachá à autoridade
    ski = x509.SubjectKeyIdentifier.from_public_key(chave_usuario.public_key())
    aki = x509.AuthorityKeyIdentifier.from_issuer_public_key(ca_privada.public_key())

    # Gera o certificado com validade de 30 dias
    certificado = x509.CertificateBuilder().subject_name(
        identidade
    ).issuer_name(
        ca_cert.subject # Assinado pela Root CA
    ).public_key(
        chave_usuario.public_key()
    ).serial_number(
        x509.random_serial_number()
    ).not_valid_before(
        datetime.datetime.utcnow()
    ).not_valid_after(
        # Usa a variável global configurada na Célula 3
        datetime.datetime.utcnow() + datetime.timedelta(days=VALIDADE_DIAS)
    ).add_extension(
        x509.BasicConstraints(ca=False, path_length=None), critical=True, # NÃO é uma autoridade
    ).add_extension(
        ski, critical=False
    ).add_extension(
        aki, critical=False
    ).add_extension(
        x509.KeyUsage( # Pode assinar documentos digitais
            digital_signature=True,
            key_cert_sign=False,
            crl_sign=False,
            key_encipherment=False,
            content_commitment=False,
            data_encipherment=False,
            key_agreement=False,
            encipher_only=False,
            decipher_only=False
        ), critical=True
    ).add_extension(
         x509.ExtendedKeyUsage([ # Pode proteger emails (requisito Adobe)
            x509.oid.ExtendedKeyUsageOID.EMAIL_PROTECTION
        ]), critical=False
    ).sign(ca_privada, hashes.SHA256(), default_backend())

    return chave_usuario, certificado
def configurar_ambiente_pkis():
    print("🔐 Inicializando Sistema de Segurança PKI...")

    # 1. Cria a Root e o Leaf
    root_key, root_cert = criar_autoridade_certificadora()
    leaf_key, leaf_cert = criar_certificado_usuario(root_key, root_cert)

    # 2. Converte para texto (PEM) para a biblioteca ler
    pem_leaf_key = leaf_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )

    # A corrente completa: Certificado do Usuário + Certificado da Autoridade
    pem_certs_chain = leaf_cert.public_bytes(serialization.Encoding.PEM) + \
                      root_cert.public_bytes(serialization.Encoding.PEM)

    pem_root_store = root_cert.public_bytes(serialization.Encoding.PEM)
    # 3. Configura o C2PA para confiar na nossa Autoridade
    print("   -> Aplicando regras de confiança...")
    oficiais = ""
    try:
        # Tenta pegar a lista oficial da internet também
        url = "https://contentcredentials.org/trust/anchors.pem"
        with urllib.request.urlopen(url) as r: oficiais = r.read().decode('utf-8')
    except: pass
    # Junta nossa autoridade local com as oficiais
    trust_completo = pem_root_store.decode('utf-8') + "\n" + oficiais

    try:
        c2pa.load_settings({"trust": {"trust_anchors": trust_completo}})
    except Exception as e:
        print(f"⚠️ Erro ao carregar settings: {e}")

    return pem_leaf_key, pem_certs_chain
# Executa ao rodar a célula
CHAVE_PRIVADA, CADEIA_CERTIFICADOS = configurar_ambiente_pkis()
print("✅ Identidade Digital criada com sucesso!")

🔐 Inicializando Sistema de Segurança PKI...
   -> Gerando Autoridade Certificadora (Root CA)...
   -> Gerando Certificado do Usuário (Leaf)...
   -> Aplicando regras de confiança...
✅ Identidade Digital criada com sucesso!


/tmp/ipython-input-2422283530.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.datetime.utcnow()
/tmp/ipython-input-2422283530.py:46: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.datetime.utcnow() + datetime.timedelta(days=365)
/tmp/ipython-input-2422283530.py:92: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.datetime.utcnow()
/tmp/ipython-input-2422283530.py:95: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-awar

In [19]:
# ==============================================================================
# CÉLULA 5: FUNÇÃO DE LEITURA (LER MANIFESTO)
# ==============================================================================
# @title 5. Função para Ler Metadados
# Esta função abre o arquivo e procura pelo "selo" C2PA escondido.
def ler_manifesto(caminho_arquivo):
    """Lê e exibe informações C2PA de um arquivo."""

    print(f"\n--- Analisando arquivo: {os.path.basename(caminho_arquivo)} ---")

    # Descobre o formato usando o mapa global
    ext = os.path.splitext(caminho_arquivo)[1].lower()
    mime = MAPA_MIMES.get(ext, "application/octet-stream")

    try:
        # Usa a ferramenta C2PA para ler
        with c2pa.Reader(mime, caminho_arquivo) as reader:
            dados = json.loads(reader.json())
            print("✅ MANIFESTO ENCONTRADO!")
            return dados # Devolve os dados para quem pediu

    except Exception:
        # Se der qualquer erro (normalmente significa que não tem C2PA),
        # apenas retornamos None (vazio) e seguimos a vida.
        print("ℹ️ Este arquivo NÃO possui manifesto C2PA.")
        return None
    except Exception as e:
        print(f"⚠️ Erro ao tentar ler: {e}")
        return None

In [20]:
# ==============================================================================
# CÉLULA 6: FUNÇÃO DE ASSINATURA (CRIAR MANIFESTO)
# ==============================================================================
# @title 6. Função para Assinar Arquivos
# Esta é a parte mais importante. Ela cria o novo arquivo com o manifesto.
# Função auxiliar interna para assinar os bytes
def _assinador_interno(data: bytes) -> bytes:
    key = serialization.load_pem_private_key(CHAVE_PRIVADA, password=None, backend=default_backend())
    return key.sign(data, ec.ECDSA(hashes.SHA256()))
def assinar_arquivo(caminho_arquivo, nome_autor=None):
    """
    Cria uma nova versão assinada do arquivo.
    Parâmetros:
      - caminho_arquivo: O arquivo original.
      - nome_autor: O nome que aparecerá como criador (Usa o Global se vazio).
    """
    # Usa a variável global configurada na Célula 3
    if not nome_autor:
        nome_autor = NOME_AUTOR_GLOBAL

    filename = os.path.basename(caminho_arquivo)
    print(f"\n✍️ Iniciando Assinatura para: {filename} (Autor: {nome_autor})")

    # 1. Define nomes e formatos
    nome_base, ext = os.path.splitext(caminho_arquivo)
    ext_lower = ext.lower()
    caminho_saida = f"{nome_base}_assinado{ext}"

    # Usa nosso mapa global para saber o MIME type correto (ex: video/mp4)
    mime = MAPA_MIMES.get(ext_lower, "application/octet-stream")
    # 2. Configura o Manifesto (O conteúdo do "Selo")
    manifesto_dados = {
        # Quem gerou este selo? (Usamos o nome do autor aqui também)
        "claim_generator": f"Google_Colab_Antigravity/{nome_autor}",

        # Detalhes técnicos do gerador
        "claim_generator_info": [{
            "name": nome_autor,
            "version": "1.0",
            # Aqui usamos a versão real do SDK que pegamos na Célula 3
            "org.contentauth.c2pa_rs": VERSAO_C2PA
        }],

        "format": mime,
        "title": filename,

        "assertions": [
            # A) Ação de Criação (O que aconteceu?)
            {
                "label": "c2pa.actions",
                "data": {
                    "actions": [{
                        "action": "c2pa.created",
                        "softwareAgent": "Google Colab com Antigravity",
                        "actors": [{"dc:title": nome_autor}],
                        "parameters": {"format": mime} # Confirma o formato
                    }]
                }
            },

            # B) Proteção contra IA (Treinamento Generativo)
            # Padrão CAWG
            {
                "label": "cawg.training-mining",
                "data": {
                    "entries": {
                        "cawg.ai_inference": { "use": "notAllowed" },
                        "cawg.ai_generative_training": { "use": "notAllowed" }
                    }
                }
            },
            # Padrão C2PA (Redundância para garantir)
            {
                "label": "c2pa.training-mining",
                "data": {
                    "entries": {
                        "c2pa.ai_inference": { "use": "notAllowed" },
                        "c2pa.ai_generative_training": { "use": "notAllowed" }
                    }
                }
            },

            # C) Mensagem de Teste/Origem
            {
                "label": "org.contentauth.test",
                "data": {"msg": f"Gerado no Google Colab por {nome_autor}"}
            }
        ]
    }
    # 3. Executa a Assinatura (Cria o arquivo novo)
    try:
        signer = c2pa.Signer.from_callback(
            callback=_assinador_interno,
            alg=c2pa.C2paSigningAlg.ES256,
            certs=CADEIA_CERTIFICADOS.decode('utf-8'), # Envia a cadeia completa
            tsa_url="http://timestamp.digicert.com"
        )

        with c2pa.Builder(manifesto_dados) as builder:
            builder.sign_file(
                source_path=caminho_arquivo,
                dest_path=caminho_saida,
                signer=signer
            )

        print(f"✨ Sucesso! Arquivo gerado: {caminho_saida}")
        return caminho_saida

    except Exception as e:
        print(f"❌ Erro Crítico na Assinatura: {e}")
        return None

In [21]:
# ==============================================================================
# CÉLULA 7: EXECUÇÃO RÁPIDA (PAINEL DE TEXTO)
# ==============================================================================
# @title 7. Painel de Controle (Texto)
# Execute para usar rápido no terminal, sem interface gráfica complexa.
print(f">>> USANDO AUTOR: {NOME_AUTOR_GLOBAL} <<<")
print("(Caso queira mudar, execute novamente a Célula 3)")
nome_user = NOME_AUTOR_GLOBAL
print("\n>>> UPLOAD DE ARQUIVOS <<<")
print("Envie imagens, vídeos ou áudios.")
enviados = files.upload()
for arquivo in enviados.keys():
    # 1. Tenta Ler
    dados_existentes = ler_manifesto(arquivo)

    if dados_existentes:
        print("📄 Documento já possui C2PA. Exibindo dados abaixo:")
        json_bonito = json.dumps(dados_existentes, indent=2, ensure_ascii=False)
        display(Markdown(f"```json\n{json_bonito}\n```"))
    else:
        # 2. Se não tem, Assina
        novo = assinar_arquivo(arquivo, nome_autor=nome_user)

        if novo:
            print("⬇️ Baixando arquivo novo...")
            files.download(novo)

            # Mostra o JSON criado
            # Precisamos ler de novo o arquivo recém criado para pegar os hashes reais
            ext = os.path.splitext(novo)[1].lower()
            mime = MAPA_MIMES.get(ext, "application/octet-stream")
            try:
                with c2pa.Reader(mime, novo) as r:
                    final_json = json.loads(r.json())
                    display(Markdown(f"```json\n{json.dumps(final_json, indent=2)}\n```"))
            except: pass

>>> USANDO AUTOR: Mário Lúcio <<<
(Caso queira mudar, execute novamente a Célula 3)

>>> UPLOAD DE ARQUIVOS <<<
Envie imagens, vídeos ou áudios.


Saving Captura de tela 2024-08-28 004915.png to Captura de tela 2024-08-28 004915.png

--- Analisando arquivo: Captura de tela 2024-08-28 004915.png ---
ℹ️ Este arquivo NÃO possui manifesto C2PA.

✍️ Iniciando Assinatura para: Captura de tela 2024-08-28 004915.png (Autor: Mário Lúcio)
✨ Sucesso! Arquivo gerado: Captura de tela 2024-08-28 004915_assinado.png
⬇️ Baixando arquivo novo...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

```json
{
  "active_manifest": "urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d",
  "manifests": {
    "urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d": {
      "claim_generator_info": [
        {
          "name": "M\u00e1rio L\u00facio",
          "version": "1.0",
          "org.contentauth.c2pa_rs": "0.67.1"
        }
      ],
      "title": "Captura de tela 2024-08-28 004915.png",
      "instance_id": "xmp:iid:d9e638a4-8b0c-4ff2-a162-f7577b8aad77",
      "thumbnail": {
        "format": "image/png",
        "identifier": "self#jumbf=/c2pa/urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d/c2pa.assertions/c2pa.thumbnail.claim"
      },
      "ingredients": [],
      "assertions": [
        {
          "label": "c2pa.actions.v2",
          "data": {
            "actions": [
              {
                "action": "c2pa.created",
                "softwareAgent": "Google Colab com Antigravity",
                "parameters": {
                  "format": "image/png"
                },
                "actors": [
                  {}
                ]
              }
            ]
          }
        },
        {
          "label": "cawg.training-mining",
          "data": {
            "entries": {
              "cawg.ai_inference": {
                "use": "notAllowed"
              },
              "cawg.ai_generative_training": {
                "use": "notAllowed"
              }
            }
          }
        },
        {
          "label": "c2pa.training-mining",
          "data": {
            "entries": {
              "c2pa.ai_inference": {
                "use": "notAllowed"
              },
              "c2pa.ai_generative_training": {
                "use": "notAllowed"
              }
            }
          }
        },
        {
          "label": "org.contentauth.test",
          "data": {
            "msg": "Gerado no Google Colab por M\u00e1rio L\u00facio"
          }
        }
      ],
      "signature_info": {
        "alg": "Es256",
        "issuer": "Google Colab",
        "common_name": "Google Colab com Antigravity",
        "cert_serial_number": "57212304894308096007222919280798243087037312309",
        "time": "2026-01-27T22:51:00+00:00"
      },
      "label": "urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d"
    }
  },
  "validation_results": {
    "activeManifest": {
      "success": [
        {
          "code": "timeStamp.validated",
          "url": "self#jumbf=/c2pa/urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d/c2pa.signature",
          "explanation": "timestamp message digest matched: DigiCert SHA256 RSA4096 Timestamp Responder 2025 1"
        },
        {
          "code": "timeStamp.trusted",
          "url": "self#jumbf=/c2pa/urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d/c2pa.signature",
          "explanation": "timestamp cert trusted: DigiCert SHA256 RSA4096 Timestamp Responder 2025 1"
        },
        {
          "code": "claimSignature.insideValidity",
          "url": "self#jumbf=/c2pa/urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d/c2pa.signature",
          "explanation": "claim signature valid"
        },
        {
          "code": "claimSignature.validated",
          "url": "self#jumbf=/c2pa/urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d/c2pa.signature",
          "explanation": "claim signature valid"
        },
        {
          "code": "assertion.hashedURI.match",
          "url": "self#jumbf=/c2pa/urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d/c2pa.assertions/c2pa.thumbnail.claim",
          "explanation": "hashed uri matched: self#jumbf=c2pa.assertions/c2pa.thumbnail.claim"
        },
        {
          "code": "assertion.hashedURI.match",
          "url": "self#jumbf=/c2pa/urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d/c2pa.assertions/c2pa.actions.v2",
          "explanation": "hashed uri matched: self#jumbf=c2pa.assertions/c2pa.actions.v2"
        },
        {
          "code": "assertion.hashedURI.match",
          "url": "self#jumbf=/c2pa/urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d/c2pa.assertions/c2pa.hash.data",
          "explanation": "hashed uri matched: self#jumbf=c2pa.assertions/c2pa.hash.data"
        },
        {
          "code": "assertion.hashedURI.match",
          "url": "self#jumbf=/c2pa/urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d/c2pa.assertions/cawg.training-mining",
          "explanation": "hashed uri matched: self#jumbf=c2pa.assertions/cawg.training-mining"
        },
        {
          "code": "assertion.hashedURI.match",
          "url": "self#jumbf=/c2pa/urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d/c2pa.assertions/c2pa.training-mining",
          "explanation": "hashed uri matched: self#jumbf=c2pa.assertions/c2pa.training-mining"
        },
        {
          "code": "assertion.hashedURI.match",
          "url": "self#jumbf=/c2pa/urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d/c2pa.assertions/org.contentauth.test",
          "explanation": "hashed uri matched: self#jumbf=c2pa.assertions/org.contentauth.test"
        },
        {
          "code": "assertion.dataHash.match",
          "url": "self#jumbf=/c2pa/urn:c2pa:64b63f8c-70c0-4666-8fff-c8ca154e4d7d/c2pa.assertions/c2pa.hash.data",
          "explanation": "data hash valid"
        }
      ],
      "informational": [],
      "failure": []
    }
  },
  "validation_state": "Valid"
}
```

In [23]:
# ==============================================================================
# CÉLULA 8: INTERFACE GRÁFICA (GRADIO APP)
# ==============================================================================
# @title 8. App Visual Completo
# Uma interface bonita para arrastar e soltar arquivos.
def processar_via_gradio(arquivo_obj, nome_autor):
    if arquivo_obj is None:
        return "⚠️ Nenhum arquivo.", None, None, None

    caminho = arquivo_obj.name

    # 1. Ler (usa a função da Célula 5)
    dados = ler_manifesto(caminho)

    if dados:
        texto_json = json.dumps(dados, indent=2, ensure_ascii=False)
        msg_status = "✅ **O arquivo já é assinado!** Veja os detalhes ao lado."
        # Retorna o botão de download visível com o arquivo atual
        return msg_status, gr.DownloadButton(label=f"📥 Baixar {os.path.basename(caminho)}", value=caminho, visible=True), dados, texto_json

    # 2. Assinar (usa a função da Célula 6)
    # Se o campo estiver vazio, usa a variável global definida na Célula 3
    if not nome_autor:
        nome_autor = NOME_AUTOR_GLOBAL

    novo_caminho = assinar_arquivo(caminho, nome_autor=nome_autor)

    if novo_caminho:
        # Ler o resultado para exibir
        dados_novos = ler_manifesto(novo_caminho)
        # ensure_ascii=False garante que "Mário" apareça como "Mário", não "M\u00e1rio"
        texto_json = json.dumps(dados_novos, indent=2, ensure_ascii=False)
        msg_status = f"🎉 **Arquivo Assinado com Sucesso!**\n\nAutor: {nome_autor}"
        # Retorna o botão de download ATIVADO e VISÍVEL
        return msg_status, gr.DownloadButton(label="📥 CLIQUE AQUI PARA BAIXAR", value=novo_caminho, visible=True), dados_novos, texto_json
    else:
        return "❌ Erro ao processar.", gr.DownloadButton(visible=False), None, None
# Layout da Interface
with gr.Blocks(title="Antigravity C2PA", theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🛡️ Gerador de Content Credentials (C2PA)")
    gr.Markdown("Proteja sua autoria e bloqueie treinamento de IA em imagens, vídeos e áudios.")

    with gr.Row():
        # Coluna da Esquerda (Entrada)
        with gr.Column():
            entrada_arq = gr.File(label="1. Arraste o Arquivo (Qualquer formato suportado)")
            entrada_nom = gr.Textbox(
                label="2. Nome do Autor",
                placeholder=f"Padrão: {NOME_AUTOR_GLOBAL} (Deixe vazio para usar)"
            )
            btn_acao = gr.Button("🚀 Processar Arquivo", variant="primary", size="lg")
            lbl_status = gr.Markdown(label="Status") # Mensagens bonitas

        # Coluna da Direita (Saída)
        with gr.Column():
            # Botão de Download (Começa Invisivel)
            btn_download = gr.DownloadButton("📥 Aguardando processamento...", visible=False, variant="primary", size="lg")

            saida_vis = gr.JSON(label="4. Visualizador de Metadados")
            saida_raw = gr.Code(label="5. Copiar Código JSON", language="json")
    # Ação do Botão
    btn_acao.click(
        fn=processar_via_gradio,
        inputs=[entrada_arq, entrada_nom],
        outputs=[lbl_status, btn_download, saida_vis, saida_raw]
    )
app.launch(share=True, debug=True)

/tmp/ipython-input-3506110500.py:39: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Antigravity C2PA", theme=gr.themes.Soft()) as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://47dc3534260e0d43fd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1139, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error


--- Analisando arquivo: prazocerto.png ---
ℹ️ Este arquivo NÃO possui manifesto C2PA.

✍️ Iniciando Assinatura para: prazocerto.png (Autor: Mário Lúcio)
✨ Sucesso! Arquivo gerado: /tmp/gradio/fe7e98cc3df4fd1da97a013999c6ca291e13b192bc41d6993165b001cbfa5f3a/prazocerto_assinado.png

--- Analisando arquivo: prazocerto_assinado.png ---
✅ MANIFESTO ENCONTRADO!
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://47dc3534260e0d43fd.gradio.live
